What is the "clipped double-Q trick"
https://docs.cleanrl.dev/rl-algorithms/sac/

## Classes

- `Actor`
- `Critics Q1 and Q2`
- `Target Critics Q1 and Q2`
-
The clipped double-Q trick means learning two Q-functions and using the smaller of their estimates:

 $$
\boxed{Q_{\text{used}}(s,a)=\min\bigl(Q_1(s,a),Q_2(s,a)\bigr).}
$$

Its purpose is to reduce overestimation of action values.

### Target Q-networks

Target Q-networks provide a slowly changing reference for training the critics. They help stabilize bootstrapping. The target Q-networks are used to train the critics.

In entropy-regularized reinforcement learning, the agent gets a bonus reward at each time step proportional to the entropy of the policy at that timestep

$$
\pi^* = \underset{\pi}{\arg \max} \mathbb{E} \left[ \sum_{t=0}^{\infty} \gamma^t \left( R(S_t,A_t,S_{t+1}) + \alpha \mathcal{H}(\pi(\cdot |S_t)\right) \right]
$$

$\mathcal{H}(\pi(\cdot |S_t)\right)$ corresponds to the entropy of the policy at that timestep. It is a bonus because it will encourage actions that increase entropy (exploration).

### Flow of SAC agent :

$$
\boxed{
\text{collect experience}
\rightarrow
\text{train critics}
\rightarrow
\text{train actor}
\rightarrow
\text{update target critics}
}
$$

reparameterization trick ????



Applying SAC to the pendulum swingup example
(https://gymnasium.farama.org/environments/classic_control/pendulum/)



In [1]:
import gymnasium as gym
import numpy as np
from torch.nn import MSELoss

env = gym.make("Pendulum-v1", render_mode="rgb_array", g=9.81)
observation, info = env.reset(seed=123, options={"x_init": np.pi/2, "y_init": 0.5})

print(observation) # Observation space : observation is [x y ω] with ω the angular velocity

print(env.action_space) # can be a number from -2 to 2 Nm (torque)

env.close()

[ 0.8403459   0.54205054 -0.44617897]
Box(-2.0, 2.0, (1,), float32)


The Actor outputs the _parameters_ of a probability distribution, from which we sample an action. (Is that what makes SAC stochastic?)

For a normal distribution : $A_t \sim $

1. Sample noise from a standard Gaussian distribution : $
\epsilon\sim\mathcal N(0,I)$

2. Scale and shift that noise $u=\mu_\theta(s)+\sigma_\theta(s)\epsilon$

3. Clip the sample space to the action space ]-2;2[ : $
a=2\tanh(u)$

`Actor.sample` samples an action from the Actor policy $
a\sim\pi_\theta(\cdot\mid s).$

In [2]:
import torch
from torch import nn

class Actor(nn.Module):
    def __init__(self,input_features=3,output_labels=2):

        self.input_features = input_features
        self.output_labels = output_labels
        super().__init__()
        # Initialise the Normal probability distribution variables. Or do we not need to do that ?
        self.mu = 0.4
        self.sigma = 1

        self.max_action = 2 # the action space is limited to ]-2;+2[ therefore we multiply the clamped action by 2 to stay in that range
        self.actor_network = nn.Sequential(
            nn.Linear(self.input_features, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, self.output_labels),
        )

        # initialise the neural network parameters randomly ?

    def sample(self, state):
        mu, log_sigma = self.forward(state)
        log_sigma = log_sigma.clamp(-20, 2)
        sigma = log_sigma.exp()

        distribution = torch.distributions.Normal(mu, sigma)

        # Equivalent to mu + sigma * torch.randn_like(mu)
        u = distribution.rsample()

        squashed = torch.tanh(u)
        scale = torch.as_tensor(
            self.max_action, dtype=mu.dtype, device=mu.device
        )
        action = squashed * scale

        # Correct the density for tanh and action scaling.
        log_prob = distribution.log_prob(u)
        log_prob -= torch.log(scale * (1 - squashed.square()) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)

        return action, log_prob

    def forward(self, states):
        # transforms a state into Gaussian parameters σ and μ
        # fully connected DNN
        # in : state = ([x y ω]) (three scalars)
        # out : mu, sigma (two scalars
        outputs = self.actor_network(states)
        return outputs[..., :1], outputs[..., 1:]


The critic takes a single tensor $(S_t, A_t) = (x_t, y_t,ω_t, A_t )$

In [3]:
class Critic(nn.Module):
    def __init__(self,input_features,):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(4, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )

        # initialise the neural network parameters randomly ?

    def forward(self, states, actions):
        inputs = torch.cat([states, actions], dim=-1)
        return self.network(inputs)


The ReplayBuffer is used to TRAIN the neural networks

In [4]:
import random
class ReplayBuffer:
    def __init__(self):
        self.replay = []

    def __len__(self):
        return len(self.replay)

    def record(self,state,action,reward,next_state,terminal=False):
        # stored at each increment.
        self.replay.append((
            np.array(state, copy=True),
            np.array(action, copy=True),
            float(reward),
            np.array(next_state, copy=True),
            bool(terminal),
        ))

    def sample(self, batch_size):
        batch = random.sample(self.replay, batch_size)
        states, actions, rewards, next_states, terminals = zip(*batch)

        return (
            torch.as_tensor(np.stack(states), dtype=torch.float32),
            torch.as_tensor(np.stack(actions), dtype=torch.float32),
            torch.as_tensor(rewards, dtype=torch.float32).reshape(-1, 1),
            torch.as_tensor(np.stack(next_states), dtype=torch.float32),
            torch.as_tensor(terminals, dtype=torch.float32).reshape(-1, 1),
        )

In [5]:
import copy

q1 = Critic()
q2 = Critic()  # Separate, independently initialized network

q1_optimizer = torch.optim.Adam(
    q1.parameters(),
    lr=3e-4,
)

q2_optimizer = torch.optim.Adam(
    q2.parameters(),
    lr=3e-4,
)

target_q1 = copy.deepcopy(q1)
target_q2 = copy.deepcopy(q2)

target_q1.requires_grad_(False)
target_q2.requires_grad_(False)

targets = [] # list of targets y from the replay buffer

buffer = ReplayBuffer()

actor = Actor()
actor_optimizer = torch.optim.Adam(
    actor.parameters(),
    lr=3e-4,
)




## 1. Critic loss: improve the value predictions
$$
L_{Q_i}
=
\frac{1}{B}\sum_{j=1}^{B}
\left(Q_i(s_j,a_j)-y_j\right)^2,
$$
where $y_j$ is the bootstrapped target calculated from the reward and target critics.

This measures how far the critic’s predictions are from their learning targets. Can be implemented using `MSE` loss in Pytorch.

## 2. Actor loss: improve action selection
$$
L_\pi
=
\frac{1}{B}\sum_{j=1}^{B}
\left[
\alpha\log\pi_\theta(\tilde a_j\mid s_j)
-\min_i Q_i(s_j,\tilde a_j)
\right].
$$
Here, $\tilde a_j$ is freshly sampled from the actor. Minimizing this loss encourages actions with high estimated value while maintaining entropy.

$$
y(r,s',d)
=
r+\gamma(1-d)
\left(
\min_{j=1,2} Q_{\phi_{\mathrm{targ},j}}(s',\tilde{a}')
-\alpha\log\pi_\theta(\tilde{a}'\mid s')
\right),
\qquad
\tilde{a}'\sim\pi_\theta(\cdot\mid s')
$$

TODO : when the action is terminal we need to RESET THE ENVIRONMENT !

d is the _terminal indicator_
$d=0$: the task continues, so include the estimated future return.
$d=1$: the task has ended, so $1-d=0$ and the target becomes simply $y=r$. **There are no future reward**

When performing a training iteration :
- sample a batch of k transitions (s,a,r,s') from the replaybuffer at random.
(use `np.random.choice`)
- Compute the target as shown in the `y(r,s',d)` formula.
(the Q function _critics_ move slowly towards the targets by minimizing $(Q_{\phi}(s,a) - y)^2$

The flow of update is as follows : 1) train the critic 2) train the actor, then 3) slowly update the target critics.

B is a batch sampled from the ReplayBuffer, and $|\textbf{B}|$ is the number of transitions in it.

Update of the actor :

For _each state_ $s \in \textbf{B}$, i.e every action in the batch : sample a fresh action $$\hat{a_\phi}(s) \sim \pi_\theta (\cdot | s) $$
$$F(\theta) = \frac{1}{|\textbf{B}|} \sum_{s \in B} \left[ \underset{i}{\min} Q_{\phi_i}(s,\hat{a_\phi}(s) - \alpha \log(\pi_\theta (\hat{a_\phi}(s) | s)) \right]$$

The actor takes a step of gradient ascent to increase this quantity:

$$
\boxed{\theta\leftarrow\theta+\eta_\pi\nabla_\theta F(\theta).}
$$


| Critic update | Actor update |
|---|---|
| Evaluates actions stored in the buffer | Evaluates freshly sampled actions |
| Changes critic parameters \(\phi_i\) | Changes actor parameters \(\theta\) |
| Reduces prediction error | Improves the value–entropy tradeoff |

Updating procedure of a `Sequential` neural network : propagation calculates gradients, then optimizer changes the weights. These are two separate phases.

1. Calculate loss: `loss = loss_function(prediction, target)`
2. Clear gradients from previous updates : `optimizer.zero_grad()`
3. Backpropagate : `loss.backward()`
4. Update weights : `optimizer.step()`

## Updating the Critics

In [6]:
mse_loss = nn.MSELoss()

def update_critics(states, actions, targets):
    # Predictions for actions stored in the replay buffer.
    q1_predictions = q1(states, actions)
    q2_predictions = q2(states, actions)

    # Compare predictions with the fixed learning targets.
    q1_loss = mse_loss(q1_predictions, targets.detach())
    q2_loss = mse_loss(q2_predictions, targets.detach())
    critic_loss = q1_loss + q2_loss

    # Clear old gradients, calculate new ones, then update weights.
    q1_optimizer.zero_grad()
    q2_optimizer.zero_grad()

    critic_loss.backward()

    q1_optimizer.step()
    q2_optimizer.step()

    return critic_loss.item()

## Updating the Actor

In [7]:
def compute_actor_loss(actor, q1, q2, states, alpha=0.9):
    # Fresh actions sampled at the replay batch's current states.
    actions, log_prob = actor.sample(states)

    q_value = torch.minimum(
        q1(states, actions),
        q2(states, actions),
    )

    return (alpha * log_prob - q_value).mean()

def update_actor(states):
    # Freeze critic parameters while retaining gradients through their inputs.
    critic_parameters = list(q1.parameters()) + list(q2.parameters())
    for parameter in critic_parameters:
        parameter.requires_grad_(False)

    actor_optimizer.zero_grad()

    actor_loss = compute_actor_loss(actor, q1, q2, states)
    actor_loss.backward()
    actor_optimizer.step()

    for parameter in critic_parameters:
        parameter.requires_grad_(True)

## Update the targets

In [8]:

def compute_critic_targets(actor, target_q1, target_q2,next_states, rewards, terminals,gamma=0.9, alpha=0.9):
    # uses data from the replay buffer
    with torch.no_grad():
        next_actions, next_log_prob = actor.sample(next_states)

        next_q = torch.minimum(
            target_q1(next_states, next_actions),
            target_q2(next_states, next_actions),
        )

        targets = rewards + gamma * (1.0 - terminals.float()) * (
            next_q - alpha * next_log_prob
        )

    return targets

def update_target_networks(q1, q2, target_q1, target_q2, tau=0.005):
    with torch.no_grad():
        for critic, target_critic in [
            (q1, target_q1),
            (q2, target_q2),
        ]:
            for parameter, target_parameter in zip(
                critic.parameters(), target_critic.parameters()
            ):
                target_parameter.mul_(1 - tau)
                target_parameter.add_(tau * parameter)

#slowly copying weights

In [9]:
# Updating function
batch_size = 5
update_spacing = 10

def its_time_to_update(gamma=0.9, alpha=0.9, tau=0.005):
    states, actions, rewards, next_states, terminals = buffer.sample(
        batch_size
    )

    # 1. Compute this batch's numerical targets y.
    targets = compute_critic_targets(
        actor, target_q1, target_q2,
        next_states, rewards, terminals,
        gamma=gamma, alpha=alpha,
    )

    # 2. Train the current critics.
    critic_loss = update_critics(states, actions, targets)

    # 3. Train the actor.
    update_actor(states)

    # 4. Slowly update the target-network weights.
    update_target_networks(q1, q2, target_q1, target_q2, tau=tau)

    return critic_loss

## Putting it all together : run the SAC agent on the pendulum environment

In [23]:
env = gym.make("Pendulum-v1", g=9.81)
state, info = env.reset(seed=123)
env.action_space.seed(123)

learning_starts = 10_000
total_steps = 1_000_000  # Example training budget, not a convergence guarantee

def train_agent(state):
    for step in range(total_steps):
        if step < learning_starts:
            action = env.action_space.sample()
        else:
            state_tensor = torch.as_tensor(
                state, dtype=torch.float32
            ).unsqueeze(0)

            with torch.no_grad():
                action_tensor, _ = actor.sample(state_tensor)

            action = action_tensor.squeeze(0).cpu().numpy()

        next_state, reward, terminated, truncated, info = env.step(action)

        buffer.record(state, action, reward, next_state, terminated)

        if step >= learning_starts and len(buffer) >= batch_size:
            critic_loss = its_time_to_update(
                gamma=0.99, alpha=0.2, tau=0.005
            )

        if terminated or truncated:
            state, info = env.reset()
        else:
            state = next_state

env.close()

In [24]:
# Save the models
torch.save(actor.state_dict(), "actor_model.pth")
torch.save(q1.state_dict(), "critic_1_model.pth")
torch.save(q2.state_dict(), "critic_2_model.pth")
torch.save(target_q2.state_dict(), "critic_target_2_model.pth")
torch.save(target_q1.state_dict(), "critic_target_1_model.pth")

To evaluate the model, we make the actor's policy _deterministic_ and disable learning :

In [17]:
def evaluate_actor(actor, episodes=20, seed=10_000, render=False):
    eval_env = gym.make(
        "Pendulum-v1",
        g=9.81,  # Match the training environment
        render_mode="human" if render else None,
    )

    device = next(actor.parameters()).device
    was_training = actor.training
    actor.eval()

    returns = []

    try:
        with torch.no_grad():
            for episode in range(episodes):
                state, info = eval_env.reset(seed=seed + episode)
                episode_return = 0.0

                while True:
                    state_tensor = torch.as_tensor(
                        state, dtype=torch.float32, device=device
                    ).unsqueeze(0)

                    mu, _ = actor(state_tensor)
                    action = torch.tanh(mu) * actor.max_action
                    action = action.squeeze(0).cpu().numpy()

                    state, reward, terminated, truncated, info = (
                        eval_env.step(action)
                    )
                    episode_return += reward

                    if terminated or truncated:
                        break

                returns.append(episode_return)
    finally:
        eval_env.close()
        actor.train(was_training)

    returns = np.asarray(returns)

    print(f"Mean episode return: {returns.mean():.1f}")
    print(f"Standard deviation:  {returns.std(ddof=1):.1f}"
          if episodes > 1 else "Standard deviation:  N/A")
    print(f"Best / worst:        {returns.max():.1f} / {returns.min():.1f}")

    return returns

In [25]:
%pip install pygame
evaluate_actor(actor, episodes=10, render=True)


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: /usr/local/bin/python3.13 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


KeyboardInterrupt: 